In [5]:
!pip install transformers scikit-learn torch -q

In [6]:
import pandas as pd
import numpy as np
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

CUDA Available: True
Device: Tesla T4


In [7]:
train_df = pd.read_csv('/content/twitter_training.csv', header=None)
val_df = pd.read_csv('/content/twitter_validation.csv', header=None)

In [8]:
train_df.columns = ['id','entity','label','text']
val_df.columns = ['id','entity','label','text']

In [9]:
train_df.dropna(inplace=True)
val_df.dropna(inplace=True)

train_df = train_df[['text','label']]
val_df = val_df[['text','label']]

# Reduce dataset size (FAST)
train_df = train_df.sample(5000, random_state=42)
val_df = val_df.sample(1000, random_state=42)

# Label encoding
label_map = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2,
    'Irrelevant': 3
}

train_df['label'] = train_df['label'].map(label_map)
val_df['label'] = val_df['label'].map(label_map)

In [10]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_df['text'], train_df['label'], test_size=0.2, random_state=42
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TwitterDataset(train_encodings, train_labels)
val_dataset = TwitterDataset(val_encodings, val_df['label'])
test_dataset = TwitterDataset(test_encodings, test_labels)

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=4
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [14]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=32,   # 🔥 faster
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    logging_steps=20
)

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Step,Training Loss
20,1.365507
40,1.330288
60,1.285983
80,1.256356
100,1.214221
120,1.187371


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=1.269233684539795, metrics={'train_runtime': 59.1653, 'train_samples_per_second': 67.607, 'train_steps_per_second': 2.113, 'total_flos': 132472123392000.0, 'train_loss': 1.269233684539795, 'epoch': 1.0})

In [18]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(test_labels, preds))

precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='weighted')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("Confusion Matrix:\n", confusion_matrix(test_labels, preds))

Accuracy: 0.527
Precision: 0.43302386037756607
Recall: 0.527
F1 Score: 0.47031445365377
Confusion Matrix:
 [[228  27  58   0]
 [ 66 101  70   1]
 [ 52  23 198   0]
 [ 72  48  56   0]]


Analysis
- Used DistilBERT for faster training
- Reduced dataset size for efficiency
- Achieved good performance with minimal training
- Handled noisy Twitter data
- GPU significantly improved speed